## BMLL Notebook Educational Series
### Market State Analysis: Auction Timing, Duration & Unscheduled Events

Exchange trading sessions follow a structured sequence of **market states**:

| State | Description |
|---|---|
| `OPENING_AUCTION` | Pre-market call period; orders collected before opening |
| `CONTINUOUS_TRADING` | Lit order book is open for continuous matching |
| `CLOSING_AUCTION` | End-of-day call period; sets the official close price |
| `INTRADAY_AUCTION` | Scheduled mid-session pause (volatility/crossing auctions) |
| `UNSCHEDULED_AUCTION` | Circuit breaker or volatility interruption outside normal schedule |

This notebook produces **monthly statistics per venue** on three dimensions:

1. **Auction duration** — how long closing and opening auctions typically last,
   and whether duration is stable month-to-month
2. **Continuous trading timing variance** — how consistently a venue opens and
   closes continuous trading (useful for detecting schedule irregularities)
3. **Unscheduled auction frequency** — a proxy for volatility regime; venues or
   months with many unscheduled auctions saw more circuit-breaker events

In [ ]:
dbutils.widgets.text("mics", "XLON,XETR,XHKG,XNYS")
dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "")

mics = dbutils.widgets.get("mics")
start_date = dbutils.widgets.get("start_date")
end_date = dbutils.widgets.get("end_date")

In [ ]:
# ── Table configuration ─────────────────────────────────────────────────
# Set USE_SAMPLE_DATA=true/false as an environment variable (or Databricks
# cluster/job env var) to switch between trial views (DD-1167) and full
# production views without changing any code.
# Override catalog names via MARKET_DATA_CATALOG / DERIVED_DATA_CATALOG.
import os as _os

_s = _os.environ.get("USE_SAMPLE_DATA", "").lower()
USE_SAMPLE_DATA = True if _s == "true" else (False if _s == "false" else False)

MARKET_DATA_CATALOG  = _os.environ.get("MARKET_DATA_CATALOG",  "market_data")
DERIVED_DATA_CATALOG = _os.environ.get("DERIVED_DATA_CATALOG", "derived_data")

if USE_SAMPLE_DATA:
    # DD-1167 trial views — 6-month rolling window, no subscription required
    REFERENCE_TBL               = f"{MARKET_DATA_CATALOG}.REF_GLOBAL_PROD"
    ANALYTICS_TBL               = f"{MARKET_DATA_CATALOG}.ANA_GLOBAL_PROD"
    TRADES_TBL                  = f"{MARKET_DATA_CATALOG}.TRDS_GLOBAL_PROD"
    TRADES_PLUS_TBL             = f"{MARKET_DATA_CATALOG}.TDSP_GLOBAL_PROD"
    L2_EQUITY_TBL               = f"{MARKET_DATA_CATALOG}.L2_GLOBAL_PROD"
    CBBO_TBL                    = f"{MARKET_DATA_CATALOG}.CBBO_GLOBAL_PROD"
    DAILY_CLASSIFIED_TRADES_TBL = f"{MARKET_DATA_CATALOG}.DCT_GLOBAL_PROD"
    # No dedicated sample views for these tables — fall back to production views
    IMBALANCE_TBL               = f"{MARKET_DATA_CATALOG}.vw_imbalance"
    MARKET_STATE_TBL            = f"{MARKET_DATA_CATALOG}.vw_market_state"
else:
    # Full production views
    REFERENCE_TBL               = f"{MARKET_DATA_CATALOG}.vw_reference"
    ANALYTICS_TBL               = f"{DERIVED_DATA_CATALOG}.vw_analytics"
    TRADES_TBL                  = f"{MARKET_DATA_CATALOG}.vw_trades"
    TRADES_PLUS_TBL             = f"{MARKET_DATA_CATALOG}.vw_trades_plus"
    L2_EQUITY_TBL               = f"{MARKET_DATA_CATALOG}.vw_l2_data_equity"
    CBBO_TBL                    = f"{MARKET_DATA_CATALOG}.vw_cbbo_equity"
    DAILY_CLASSIFIED_TRADES_TBL = f"{DERIVED_DATA_CATALOG}.vw_daily_classified_trades"
    IMBALANCE_TBL               = f"{MARKET_DATA_CATALOG}.vw_imbalance"
    MARKET_STATE_TBL            = f"{MARKET_DATA_CATALOG}.vw_market_state"

In [ ]:
import datetime as _dt
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd

# ── Rolling date-range guard ──────────────────────────────────────────────────
# The BMLL sample data covers a rolling 6-month window ending today.
# Reset any date range that falls entirely before that window so the notebook
# always runs correctly on both trial and production datasets.
_today = _dt.date.today()
_window_start = _today - _dt.timedelta(days=182)

if not start_date:
    start_date = (_today - _dt.timedelta(days=90)).isoformat()
if not end_date:
    end_date = _today.isoformat()

_sd = _dt.date.fromisoformat(start_date)
_ed = _dt.date.fromisoformat(end_date)

if _ed < _window_start:
    end_date   = _today.isoformat()
    start_date = (_today - _dt.timedelta(days=90)).isoformat()
    print(f"Date range outside 6-month sample window — reset to {start_date} / {end_date}")
elif _sd < _window_start:
    start_date = _window_start.isoformat()

print(f"Analysis period: {start_date} → {end_date}")

### Step 1: Load Market State Transitions

The MARKET_STATE table records every state-change event per listing per day.
Each row marks the **entry** into a new state; duration is computed using a
`LEAD` window function to find when the next transition occurs.

We filter to states of interest and work at the MIC level by taking the
**median** start time and duration across all listings on the same venue —
this removes outliers from suspended or halted securities.

In [ ]:
mic_list = ", ".join(f"'{m.strip()}'" for m in mics.split(","))

# Force UTC so that from_unixtime() and HOUR/MINUTE/SECOND are DST-free.
# All start_minutes values are therefore UTC-based; venues that trade in other
# timezones will show their local offset as a constant shift, but that shift
# is stable — it never inflates the variance the way DST transitions would.
spark.conf.set("spark.sql.session.timeZone", "UTC")

sql = f"""
WITH state_events AS (
    SELECT
        TRADE_DATE,
        MIC,
        LISTING_ID,
        MARKET_STATE,
        BMLL_SEQUENCE_NO,
        from_unixtime(CAST(TIMESTAMP_NANOSECONDS AS DOUBLE) / 1e9) AS event_ts,
        LEAD(from_unixtime(CAST(TIMESTAMP_NANOSECONDS AS DOUBLE) / 1e9))
            OVER (PARTITION BY LISTING_ID, TRADE_DATE
                  ORDER BY TIMESTAMP_NANOSECONDS, BMLL_SEQUENCE_NO) AS next_ts
    FROM {MARKET_STATE_TBL}
    WHERE TRADE_DATE BETWEEN '{start_date}' AND '{end_date}'
      AND MIC IN ({mic_list})
      AND MARKET_STATE IN (
          'OPENING_AUCTION',
          'CONTINUOUS_TRADING',
          'CLOSING_AUCTION',
          'INTRADAY_AUCTION',
          'UNSCHEDULED_AUCTION'
      )
),
durations AS (
    SELECT
        TRADE_DATE,
        MIC,
        LISTING_ID,
        MARKET_STATE,
        event_ts,
        UNIX_TIMESTAMP(next_ts) - UNIX_TIMESTAMP(event_ts) AS duration_seconds,
        -- Time-of-day in minutes since 00:00 UTC (for timing variance analysis)
        HOUR(event_ts) * 60.0 + MINUTE(event_ts) + SECOND(event_ts) / 60.0 AS start_minutes
    FROM state_events
    WHERE next_ts IS NOT NULL
      AND UNIX_TIMESTAMP(next_ts) - UNIX_TIMESTAMP(event_ts) > 0
      -- Exclude implausibly long durations (>4 hours = state carried over overnight)
      AND UNIX_TIMESTAMP(next_ts) - UNIX_TIMESTAMP(event_ts) < 14400
)
SELECT
    YEAR(TRADE_DATE)  AS year,
    MONTH(TRADE_DATE) AS month,
    MIC,
    MARKET_STATE,
    COUNT(DISTINCT TRADE_DATE) AS trading_days,
    COUNT(*)                   AS n_events,
    PERCENTILE_APPROX(duration_seconds, 0.5)  AS median_duration_sec,
    AVG(duration_seconds)                     AS avg_duration_sec,
    STDDEV(duration_seconds)                  AS std_duration_sec,
    PERCENTILE_APPROX(duration_seconds, 0.25) AS p25_duration_sec,
    PERCENTILE_APPROX(duration_seconds, 0.75) AS p75_duration_sec,
    PERCENTILE_APPROX(start_minutes, 0.5)     AS median_start_minutes,
    STDDEV(start_minutes)                     AS std_start_minutes
FROM durations
GROUP BY 1, 2, 3, 4
ORDER BY 1, 2, 3, 4
"""

df_stats = spark.sql(sql).toPandas()

print(f"Loaded {len(df_stats):,} monthly metric rows")
print(f"MICs: {sorted(df_stats['MIC'].unique())}")
print(f"States: {sorted(df_stats['MARKET_STATE'].unique())}")

In [ ]:
# Separate datasets by state type
closing_stats = df_stats[df_stats["MARKET_STATE"] == "CLOSING_AUCTION"].copy()
opening_stats = df_stats[df_stats["MARKET_STATE"] == "OPENING_AUCTION"].copy()
continuous_stats = df_stats[df_stats["MARKET_STATE"] == "CONTINUOUS_TRADING"].copy()
unscheduled_stats = df_stats[df_stats["MARKET_STATE"] == "UNSCHEDULED_AUCTION"].copy()

# Build a year-month string for axis labels
for d in [closing_stats, opening_stats, continuous_stats, unscheduled_stats]:
    d["ym"] = d["year"].astype(str) + "-" + d["month"].astype(str).str.zfill(2)

mic_colors = plt.cm.tab10.colors
mics_list = sorted(df_stats["MIC"].unique())

### Analysis 1: Closing Auction Duration by Venue (Monthly)

Closing auction duration spans from when the exchange enters the closing call period
to when it transitions to post-trade or halted state.

Most major exchanges target a fixed auction window (e.g., 5 minutes on LSE),
but actual duration can vary due to:
- Price collar violations extending the auction
- Technical issues causing delayed openings
- Market-wide stress events triggering extended call periods

In [ ]:
if not closing_stats.empty:
    fig, ax = plt.subplots(figsize=(13, 5))

    for i, mic in enumerate(mics_list):
        sub = closing_stats[closing_stats["MIC"] == mic].sort_values("ym")
        if sub.empty:
            continue
        color = mic_colors[i % len(mic_colors)]
        ax.plot(sub["ym"], sub["median_duration_sec"] / 60, marker="o", linewidth=1.5,
                color=color, label=mic)
        ax.fill_between(
            sub["ym"],
            sub["p25_duration_sec"] / 60,
            sub["p75_duration_sec"] / 60,
            alpha=0.15, color=color,
        )

    ax.set_xlabel("Month")
    ax.set_ylabel("Duration (minutes)")
    ax.set_title(f"Closing Auction Duration — Median ± IQR\n{start_date} – {end_date}")
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
    ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left")
    plt.tight_layout()
    plt.show()
else:
    print("No closing auction data found.")

### Analysis 2: Opening Auction Duration by Venue (Monthly)

In [ ]:
if not opening_stats.empty:
    fig, ax = plt.subplots(figsize=(13, 5))

    for i, mic in enumerate(mics_list):
        sub = opening_stats[opening_stats["MIC"] == mic].sort_values("ym")
        if sub.empty:
            continue
        color = mic_colors[i % len(mic_colors)]
        ax.plot(sub["ym"], sub["median_duration_sec"] / 60, marker="o", linewidth=1.5,
                color=color, label=mic)
        ax.fill_between(
            sub["ym"],
            sub["p25_duration_sec"] / 60,
            sub["p75_duration_sec"] / 60,
            alpha=0.15, color=color,
        )

    ax.set_xlabel("Month")
    ax.set_ylabel("Duration (minutes)")
    ax.set_title(f"Opening Auction Duration — Median ± IQR\n{start_date} – {end_date}")
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
    ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

### Analysis 3: Continuous Trading Timing Variance

We measure when each venue **begins continuous trading** (the transition out of
the opening auction) and when it **enters the closing auction** (the transition
out of continuous trading).

All timestamps are in **UTC** to avoid DST distortions: a clocks-forward event
would otherwise appear as a one-hour schedule shift and artificially inflate
variance for that month. In UTC, each venue's schedule appears at a fixed UTC
offset (e.g., XLON opens ~08:00 UTC in winter and ~07:00 UTC in summer), but
that offset is stable within a month and does not inflate the standard deviation.

Low variance (tight distribution) means a venue runs on a consistent schedule.
High variance may indicate frequent auction extensions, volatility suspensions,
or operational incidents that push the schedule back.

The standard deviation is expressed in **minutes**: values below 1 minute indicate
a very reliable schedule; values above 5 minutes suggest meaningful variability.

In [ ]:
if not continuous_stats.empty:
    # Continuous trading START timing variance (= when continuous trading begins)
    ct_start = continuous_stats.copy()

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    # Left: Start time variance
    for i, mic in enumerate(mics_list):
        sub = ct_start[ct_start["MIC"] == mic].sort_values("ym")
        if sub.empty:
            continue
        axes[0].plot(sub["ym"], sub["std_start_minutes"], marker="o", linewidth=1.5,
                     color=mic_colors[i % len(mic_colors)], label=mic)

    axes[0].set_xlabel("Month")
    axes[0].set_ylabel("Std dev of start time (minutes)")
    axes[0].set_title("Continuous Trading Start Time Variance")
    plt.setp(axes[0].get_xticklabels(), rotation=30, ha="right")
    axes[0].legend()

    # Right: Closing auction start time variance (= when continuous trading ends)
    for i, mic in enumerate(mics_list):
        sub = closing_stats[closing_stats["MIC"] == mic].sort_values("ym")
        if sub.empty:
            continue
        axes[1].plot(sub["ym"], sub["std_start_minutes"], marker="o", linewidth=1.5,
                     color=mic_colors[i % len(mic_colors)], label=mic)

    axes[1].set_xlabel("Month")
    axes[1].set_ylabel("Std dev of start time (minutes)")
    axes[1].set_title("Closing Auction Start Time Variance")
    plt.setp(axes[1].get_xticklabels(), rotation=30, ha="right")
    axes[1].legend()

    fig.suptitle(
        f"Trading Schedule Consistency by Venue\n{start_date} – {end_date}",
        fontsize=13, fontweight="bold", y=1.01,
    )
    plt.tight_layout()
    plt.show()

### Analysis 4: Unscheduled Auction Frequency per Venue

**Unscheduled auctions** are volatility interruptions triggered by circuit breakers
when a security's price moves too sharply relative to its reference price.

Their frequency is a useful **volatility regime indicator**:
- High counts during a month → elevated intraday volatility across the venue
- Venue-specific spikes → may reflect index rebalancing, earnings season, or macro events
- Cross-venue correlated spikes → systemic stress (e.g., flash crashes, macro shocks)

Note: counts are the number of *listing-level* interruptions (one volatile stock = one event).

In [ ]:
if not unscheduled_stats.empty:
    pivot = unscheduled_stats.pivot_table(
        index="ym", columns="MIC", values="n_events", aggfunc="sum", fill_value=0
    ).sort_index()

    fig, ax = plt.subplots(figsize=(13, 5))
    pivot.plot(kind="bar", ax=ax, colormap="tab10", width=0.75)

    ax.set_xlabel("Month")
    ax.set_ylabel("Unscheduled auction events")
    ax.set_title(
        f"Unscheduled Auction Frequency per Venue (Monthly)\n{start_date} – {end_date}"
    )
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
    ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", title="MIC")
    plt.tight_layout()
    plt.show()

    print("\nTotal unscheduled auctions by MIC:")
    print(unscheduled_stats.groupby("MIC")["n_events"].sum().sort_values(ascending=False).to_string())
else:
    print("No unscheduled auction events found in the selected period.")

### Summary Table

Full monthly statistics across all states and venues.

In [ ]:
summary = df_stats.copy()
summary["median_duration_min"] = (summary["median_duration_sec"] / 60).round(1)
summary["std_duration_min"] = (summary["std_duration_sec"] / 60).round(2)
summary["std_start_min"] = summary["std_start_minutes"].round(2)

display_cols = ["year", "month", "MIC", "MARKET_STATE",
                "trading_days", "n_events",
                "median_duration_min", "std_duration_min",
                "std_start_min"]

print(summary[display_cols]
      .sort_values(["year", "month", "MIC", "MARKET_STATE"])
      .to_string(index=False))